# BBO capstone — Weeks 1–12: data, progress and query selection

**Reconstructed teaching/reference notebook**, not the original historical notebook. Uses the recorded initial observations and the 12 submitted queries per function. Run from the repository root.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.optimisation import read_function_data, propose_query
print("Repository root:", ROOT)


## Dataset overview
The table below shows dataset sizes and the strongest observed value for each function. The unknown global optimum is not provided.

In [ ]:
summary = pd.read_json(ROOT / "data/processed/summary.json")
display(summary)
weekly = pd.read_csv(ROOT / "data/processed/query_history.csv")
assert len(weekly) == 12 * 8
assert weekly.groupby("function").size().eq(12).all()


## Best-so-far curves
Function outputs have very different scales, so they are plotted separately.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for f, ax in zip(range(1, 9), axes.flat):
    X, y = read_function_data(ROOT, f)
    starter_n = int(summary.loc[summary.function.eq(f), "initial_count"].iloc[0])
    recent = y[starter_n:]
    running = np.maximum.accumulate(np.r_[np.max(y[:starter_n]), recent])[1:]
    ax.plot(np.arange(1, 13), running, marker="o", linewidth=1.5)
    ax.set_title(f"Function {f}")
    ax.set_xlabel("Week")
    ax.set_ylabel("Best observed")
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


## Worked example: propose an additional candidate
The next cell fits an ARD Matérn Gaussian Process to **Function 4** and maximises Expected Improvement over Sobol and local candidates. This is a demonstration only, not a historical submission or an actual portal evaluation. It can take some time to fit.

In [ ]:
X, y = read_function_data(ROOT, 4)
candidate, diagnostics = propose_query(X, y, seed=42, n_candidates=2048)
print("Suggested candidate (not evaluated):", "-".join(f"{v:.6f}" for v in candidate))
print("Expected Improvement score:", diagnostics["acquisition"])
print("Learned kernel:", diagnostics["kernel"])


## Limitations and provenance
Historical decisions included manual review and other changing heuristics, so this demo need not reproduce the prior 12 rounds. No true objective formula is available to evaluate a newly proposed candidate. Refer to `docs/datasheet.md`, `docs/model_card.md`, and `docs/methodology.md`.